# Huấn luyện mô hình URL phishing trên Colab GPU

Notebook tải **PhishVN v4**, giữ nguyên train/validation/test theo tên miền và thời gian,
huấn luyện **Random Forest, LightGBM, XGBoost CUDA** và **fine-tune DomURLs_BERT**.
Logistic Regression và Random Forest sử dụng cấu hình tham chiếu của PhishVN để so sánh.

**Chạy:** Colab → Runtime → Change runtime type → GPU → chạy các cell theo thứ tự.
Không cần API key hoặc tài khoản Kaggle/Hugging Face. Nếu dùng máy Jupyter riêng có sẵn
PyTorch CUDA, giữ môi trường đó và chạy cell cài thư viện. Với A100/H100, batch BERT tự chọn theo VRAM.

- Đổi `RUN_BERT=False` nếu chỉ muốn phần ML; `RUN_TREES=False` nếu chỉ muốn fine-tune encoder.
- Mặc định 3 seed cho ML, 1 seed cho BERT; có thể tăng `BERT_SEEDS` để dùng GPU chạy nhiều lần.
- `SMOKE_TEST=True` chỉ kiểm tra mã bằng ít mẫu, **không dùng chỉ số đó cho báo cáo**.
- Bật `USE_DRIVE=True` nếu muốn checkpoint tồn tại sau khi phiên Colab ngắt.

PhishVN có các nhãn từ nhiều nguồn và tầng gold/silver/bronze. Nhãn dương bao gồm cả
URL lừa đảo khác ngoài đánh cắp thông tin đăng nhập. Notebook báo cáo riêng các tầng,
không coi kết quả trên bộ này là chất lượng trên mọi URL ngoài thực tế.

In [ ]:
# 1. Cài thư viện. Giữ PyTorch/CUDA hiện có, không cài lại torch từ PyPI.
import subprocess, sys, importlib.metadata
try:
    importlib.metadata.version("torch")
except importlib.metadata.PackageNotFoundError as exc:
    raise RuntimeError("Cần môi trường đã có PyTorch CUDA: chọn GPU trong Colab hoặc cài PyTorch theo pytorch.org.") from exc
packages = [
    "transformers==4.57.6", "accelerate==1.12.0", "datasets==4.4.2",
    "xgboost==3.0.5", "lightgbm==4.6.0", "tldextract==5.3.0",
    "scikit-learn==1.7.2", "pandas>=2.2,<3", "numpy>=1.26,<3",
    "matplotlib>=3.8,<4", "requests>=2.31,<3", "safetensors>=0.4.5", "joblib>=1.3",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
expected_loaded={"sklearn":"1.7.2","transformers":"4.57.6","xgboost":"3.0.5",
                 "lightgbm":"4.6.0","datasets":"4.4.2","tldextract":"5.3.0"}
stale=[name for name,version in expected_loaded.items()
       if name in sys.modules and getattr(sys.modules[name],"__version__",None)!=version]
if stale:
    raise RuntimeError("Đã cài phiên bản mới nhưng bộ nhớ còn bản cũ: "+str(stale)+
        ". Chọn Runtime → Restart session, rồi chạy từ Cấu hình trở đi.")
print("Nếu Colab yêu cầu restart, hãy restart session rồi chạy từ Cấu hình trở đi.")

## 2. Cấu hình

Mặc định chạy cả ML và BERT. BERT fine-tune toàn bộ khoảng 110 triệu tham số;
không cần LoRA hoặc lượng tử hóa với GPU mạnh. DomURLs_BERT là encoder tiền huấn luyện,
notebook thêm lớp phân loại mới trước khi fine-tune.

`DATA_SCOPE="all"` dùng toàn bộ các tầng. `verified_core` chỉ dùng gold/silver,
nhưng validation ở chế độ đó chỉ có 24 mẫu dương trước xử lý nên ngưỡng kém ổn định.

In [ ]:
CFG = {
    "RUN_TREES": True,
    "RUN_BERT": True,
    "ML_SEEDS": [42, 2026, 3407],
    "BERT_SEEDS": [42],               # đổi thành [42, 2026, 3407] để chạy 3 seed BERT
    "SEED": 42,
    "SMOKE_TEST": False,
    "ALLOW_CPU": False,
    "USE_DRIVE": False,
    "WORK_DIR": "/content/url_phishing_training",
    "DRIVE_DIR": "/content/drive/MyDrive/url_phishing_training",
    "HF_CACHE_DIR": "/content/url_model_cache",
    "LOCAL_DATA_ZIP": "",             # trống: tự tải; hoặc đường dẫn ZIP đúng phiên bản
    "DATASET_ID": "b97hxbxtpd", "DATASET_VERSION": 4,
    "DATA_ARCHIVE": "PhishVN_v3.1.0_open.zip",
    "DATA_SHA256": "308351e9d0c0fca13a81f2f63524a7dee08ccba6c775848d125b3da5525d0ab5",
    "DATA_SCOPE": "all",
    "CPU_THREADS": 8,
    "BOOST_ROUNDS": 1800,
    "EARLY_STOPPING": 100,
    "MIN_WARNING_PRECISION": 0.80,
    "MIN_HIGH_PRECISION": 0.95,
    "MIN_HIGH_COUNT": 30,
    "MODEL_ID": "amahdaouy/DomURLs_BERT",
    "MODEL_REVISION": "ba5fa28ae1361f8115630e252ac49535ffe3bab1",
    "BERT_MAX_LENGTH": 256,
    "BERT_EPOCHS": 4,
    "BERT_LR": 2e-5,
    "BERT_BATCH_SIZE": 0,             # 0: tự chọn theo VRAM; có thể đặt 64/128
    "BERT_EFFECTIVE_BATCH": 64,
    "BERT_GRAD_CHECKPOINTING": None,   # None: tự chọn; True: tiết kiệm VRAM
    "RESUME_TRAINING": True,
}
assert CFG["RUN_TREES"] or CFG["RUN_BERT"]
assert CFG["DATA_SCOPE"] in {"all", "verified_core"}
assert 0 < CFG["MIN_WARNING_PRECISION"] <= CFG["MIN_HIGH_PRECISION"] <= 1

In [ ]:
import os, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["WANDB_DISABLED"] = "true"
import gc, hashlib, importlib.metadata, itertools, json, math, re, shutil, stat, time, zipfile
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import requests
import torch
import tldextract
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb
import lightgbm as lgb
from IPython.display import display
from datasets import Dataset, DatasetDict
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix,
    classification_report, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (AutoConfig, AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments, EarlyStoppingCallback, set_seed)
from transformers.trainer_utils import get_last_checkpoint

HAS_CUDA=torch.cuda.is_available()
if not HAS_CUDA and not CFG["ALLOW_CPU"]:
    raise RuntimeError("Chưa có GPU CUDA. Chọn GPU trong Colab hoặc kiểm tra PyTorch CUDA/driver.")
if CFG["USE_DRIVE"]:
    from google.colab import drive
    drive.mount("/content/drive")
ROOT=Path(CFG["DRIVE_DIR"] if CFG["USE_DRIVE"] else CFG["WORK_DIR"])
DATA_DIR=ROOT/"data"; DATA_DIR.mkdir(parents=True,exist_ok=True)
VRAM_GB=torch.cuda.get_device_properties(0).total_memory/1024**3 if HAS_CUDA else 0
if HAS_CUDA:
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32=True
set_seed(CFG["SEED"])
VERSIONS={p:importlib.metadata.version(p) for p in ["torch","transformers","tokenizers",
    "datasets","accelerate","xgboost","lightgbm","scikit-learn","pandas","numpy","tldextract"]}
HARDWARE={"gpu":torch.cuda.get_device_name(0) if HAS_CUDA else None,
          "vram_gb":VRAM_GB,"torch_cuda":torch.version.cuda,"cpu_threads":CFG["CPU_THREADS"]}
print("Thiết bị:",HARDWARE)
print("XGBoost:","CUDA" if HAS_CUDA else "CPU (được cho phép)","| RF/LightGBM: CPU")
print(json.dumps(VERSIONS,indent=2))

## 3. Tải và kiểm tra dữ liệu

Nguồn: [PhishVN V4](https://data.mendeley.com/datasets/b97hxbxtpd/4), DOI `10.17632/b97hxbxtpd.4`.
Notebook kiểm tra SHA-256 của ZIP và các tập chia. Nếu máy chủ chặn tải tự động,
tải `PhishVN_v3.1.0_open.zip` từ trang trên, đưa vào Colab và đặt `LOCAL_DATA_ZIP`.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def download_phishvn():
    local = CFG["LOCAL_DATA_ZIP"].strip()
    archive = Path(local) if local else DATA_DIR / CFG["DATA_ARCHIVE"]
    if not archive.is_file():
        session = requests.Session()
        session.headers.update({"User-Agent": "Mozilla/5.0"})
        metadata_url = (
            f'https://data.mendeley.com/public-api/datasets/{CFG["DATASET_ID"]}/files'
        )
        try:
            response = session.get(metadata_url, params={
                "folder_id": "root", "version": CFG["DATASET_VERSION"],
            }, timeout=(15, 60))
            response.raise_for_status()
            files = response.json()
            matching = [x for x in files if x.get("filename") == CFG["DATA_ARCHIVE"]]
            if len(matching) != 1:
                raise ValueError("Không xác định được gói PhishVN đúng phiên bản.")
            selected = matching[0]
            details = selected["content_details"]
            if details.get("sha256_hash") != CFG["DATA_SHA256"]:
                raise ValueError("SHA-256 của phiên bản trên máy chủ đã thay đổi; cần kiểm tra nguồn.")
            target = archive.with_suffix(".part")
            with session.get(details["download_url"], stream=True, timeout=(15, 120)) as r:
                r.raise_for_status()
                with target.open("wb") as stream:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            stream.write(chunk)
            target.replace(archive)
            (DATA_DIR / "download_metadata.json").write_text(
                json.dumps(selected, indent=2), encoding="utf-8")
        except Exception as exc:
            raise RuntimeError(
                "Chưa tải được dữ liệu. Tải PhishVN_v3.1.0_open.zip từ DOI "
                "https://data.mendeley.com/datasets/b97hxbxtpd/4, "
                "đưa ZIP lên Colab, điền LOCAL_DATA_ZIP rồi chạy lại cell này. "
                f"Chi tiết: {exc}"
            ) from exc
    actual = sha256_file(archive)
    if actual != CFG["DATA_SHA256"]:
        raise ValueError(f"ZIP sai phiên bản hoặc tải chưa đủ. SHA-256 thực tế: {actual}")
    unpacked = DATA_DIR / "phishvn_v4"
    unpacked.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as package:
        for item in package.infolist():
            member = Path(item.filename)
            if member.is_absolute() or ".." in member.parts or "\\" in item.filename:
                raise ValueError("Đường dẫn trong ZIP không hợp lệ.")
            if stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError("ZIP có symbolic link.")
        if sum(i.file_size for i in package.infolist()) > 300 * 1024 * 1024:
            raise ValueError("Gói giải nén lớn bất thường.")
        package.extractall(unpacked)
    manifest = (unpacked / "MANIFEST.txt").read_text(encoding="utf-8")
    for split in ["train", "val", "test"]:
        relative = f"data/splits/url_{split}.csv"
        match = re.search(r"(?m)^([0-9a-f]{64})\s+" + re.escape(relative) + r"\s", manifest)
        if match is None or sha256_file(unpacked / relative) != match.group(1):
            raise ValueError(f"Tập {split} không khớp MANIFEST.")
    print("Đã xác minh:", archive.name, "| SHA-256:", actual)
    return unpacked


PHISHVN_DIR = download_phishvn()
frames_raw = {split: pd.read_csv(
    PHISHVN_DIR / f"data/splits/url_{split}.csv",
    dtype=str, keep_default_na=False,
) for split in ["train", "val", "test"]}
display(pd.DataFrame([
    {"split": name, "rows": len(df), **df["label"].value_counts().to_dict()}
    for name, df in frames_raw.items()
]))
display(frames_raw["train"].groupby(["source", "tier", "label"]).size().rename("rows").reset_index())

## 4. Làm sạch và kiểm tra rò rỉ dữ liệu

Giữ nguyên các tập chia gốc. Loại URL lỗi/trùng trong từng tập, dừng nếu nhãn mâu thuẫn
hoặc tên miền cơ sở giao nhau giữa các tập. Giữ query, fragment và chữ hoa/thường trong đường dẫn.
ML chỉ nhận đặc trưng từ URL; BERT chỉ nhận chuỗi URL. `source`, `tier`, ngày thu thập,
nhãn và ID chỉ dùng để kiểm tra, phân tích lỗi.

Phép chia PhishVN là **kết hợp tên miền/thời gian**: nhóm có thời điểm được xếp theo thời gian,
nhóm thiếu thời điểm được chia bằng hash. Không gọi toàn bộ test là dữ liệu thuần tương lai.
Các URL không được mở để lấy nội dung trang.

In [ ]:
PREPROCESSING_SOURCE = r'''
import ipaddress
from urllib.parse import urlsplit

PREPROCESS_VERSION = "domurls-urlparts-v1"

def parse_url_parts(url):
    if not isinstance(url, str) or not url.strip():
        raise ValueError("URL rỗng hoặc không phải chuỗi.")
    raw = url.strip()
    if any(c in raw for c in "\r\n\t"):
        raise ValueError("URL chứa ký tự điều khiển.")
    lower = raw.lower()
    if lower.startswith(("http://", "https://")):
        parts = urlsplit(raw)
    elif raw.startswith("//"):
        parts = urlsplit("https:" + raw)
    elif "://" not in raw:
        parts = urlsplit("https://" + raw)
    else:
        raise ValueError("Chỉ nhận URL HTTP/HTTPS hoặc chuỗi không có giao thức.")
    host = parts.hostname
    if not host or any(c.isspace() for c in parts.netloc):
        raise ValueError("Không tách được tên miền URL.")
    _ = parts.port
    return parts, host.lower().rstrip(".")

def format_url(url):
    parts, host = parse_url_parts(url)
    prefix = "[DOMAIN]"
    try:
        prefix = "[IP]" if ipaddress.ip_address(host).version == 4 else "[IPv6]"
    except ValueError:
        pass
    suffix = parts.path
    if parts.query:
        suffix += "?" + parts.query
    if parts.fragment:
        suffix += "#" + parts.fragment
    text = f"{prefix} {parts.netloc}"
    if suffix:
        text += " [PATH] " + suffix
    return text
'''
exec(PREPROCESSING_SOURCE)

extract_domain = tldextract.TLDExtract(suffix_list_urls=(), include_psl_private_domains=False)


def domain_group(url):
    _, host = parse_url_parts(url)
    try:
        host = host.encode("idna").decode("ascii")
    except UnicodeError:
        pass
    result = extract_domain(host)
    return result.top_domain_under_public_suffix or host


LABEL_TO_ID = {"benign": 0, "phishing": 1}
ID_TO_LABEL = {0: "BENIGN", 1: "PHISHING"}
cleaning_log = []
frames = {}
for split, raw in frames_raw.items():
    required = {"id", "url", "label", "tier", "source", "split"}
    if not required.issubset(raw.columns):
        raise ValueError(f"Thiếu cột ở {split}: {required - set(raw.columns)}")
    if set(raw["split"].str.strip()) != {split}:
        raise ValueError(f"Sai nhãn tập chia trong file {split}.")
    df = raw.copy()
    if CFG["DATA_SCOPE"] == "verified_core":
        df = df[df["tier"].str.lower().isin(["gold", "silver"])].copy()
    before = len(df)
    invalid = []
    prepared = []
    for row in df.to_dict("records"):
        label = row["label"].strip().lower()
        if label not in LABEL_TO_ID:
            raise ValueError(f"Nhãn chưa biết: {label!r}. Không tự quy đổi nhãn khác thành benign.")
        try:
            text = format_url(row["url"])
            group = domain_group(row["url"])
        except (ValueError, UnicodeError) as exc:
            invalid.append({"id": row["id"], "reason": str(exc)})
            continue
        row.update(labels=LABEL_TO_ID[label], model_input=text, domain_group=group)
        prepared.append(row)
    df = pd.DataFrame(prepared)
    if df.empty or set(df["labels"]) != {0, 1}:
        raise ValueError(f"Tập {split} không còn đủ hai lớp.")
    duplicated = int(df.duplicated("model_input").sum())
    df = df.drop_duplicates(["model_input", "labels"]).reset_index(drop=True)
    if df.groupby("model_input")["labels"].nunique().gt(1).any():
        raise ValueError(f"Tập {split} có URL trùng nhưng nhãn mâu thuẫn.")
    frames[split] = df
    cleaning_log.append({"split": split, "after_scope_filter": before,
                         "invalid_url": len(invalid), "duplicates": duplicated, "final_rows": len(df)})
    if invalid:
        pd.DataFrame(invalid).to_csv(DATA_DIR / f"invalid_{split}.csv", index=False)

for first, second in itertools.combinations(frames, 2):
    for column in ["id", "model_input", "domain_group"]:
        overlap = set(frames[first][column]) & set(frames[second][column])
        if overlap:
            raise ValueError(f"Phát hiện {len(overlap)} giá trị {column} trùng giữa {first}/{second}.")

ALL_PREPARED = {name: df.copy() for name, df in frames.items()}
if CFG["SMOKE_TEST"]:
    for name, limit in [("train", 64), ("val", 32), ("test", 32)]:
        df = frames[name]
        if len(df) > limit:
            _, selected = train_test_split(df, test_size=limit, stratify=df["labels"], random_state=CFG["SEED"])
            frames[name] = selected.reset_index(drop=True)
    print("CHỈ KIỂM TRA LUỒNG: dùng ít bản ghi và 2 bước train, không dùng các chỉ số này cho báo cáo.")

display(pd.DataFrame(cleaning_log))
display(pd.DataFrame([
    {"split": name, "rows": len(df), "benign": int((df.labels == 0).sum()),
     "phishing": int((df.labels == 1).sum()), "domains": df.domain_group.nunique()}
    for name, df in frames.items()
]))

In [ ]:
time_audit=[]
for split,df in ALL_PREPARED.items():
    raw_dates=df["collected_at"].str.strip()
    dates=pd.Series(pd.NaT,index=df.index,dtype="datetime64[ns, UTC]")
    ddmm=raw_dates.str.fullmatch(r"[0-9]{1,2}/[0-9]{1,2}/[0-9]{4}")
    iso=raw_dates.str.match(r"^[0-9]{4}-[0-9]{2}-[0-9]{2}")
    dates.loc[ddmm]=pd.to_datetime(raw_dates[ddmm],format="%d/%m/%Y",errors="coerce",utc=True)
    dates.loc[iso]=pd.to_datetime(raw_dates[iso],format="ISO8601",errors="coerce",utc=True)
    time_audit.append({"split":split,"rows":len(df),"dated_rows":int(dates.notna().sum()),
        "unparsed_nonempty":int((raw_dates.ne("") & dates.isna()).sum()),
        "min_collected_at":str(dates.min()),"max_collected_at":str(dates.max())})
display(pd.DataFrame(time_audit))
print("Nhãn: 0 = BENIGN (bình thường theo nhãn nguồn), 1 = PHISHING/SCAM.")

## 5. Sinh đặc trưng URL

Dùng 21 đặc trưng lexical của schema CompPhish trong mã PhishVN: độ dài, số ký tự,
tỷ lệ chữ/số/ký tự đặc biệt, độ sâu tên miền phụ, entropy, đường dẫn và query.
**Không dùng `is_https`** vì đó là dấu vết thu thập của PhishVN; các độ dài/số lượng cũng
được tính sau khi bỏ giao thức. Không gọi WHOIS/DNS để notebook và ứng dụng dùng cùng đầu vào.

Hàm bên dưới được điều chỉnh từ mã PhishVN, release `dib-113195`, giấy phép MIT;
giữ cách tính của tác giả để đối chiếu baseline. File giấy phép và nguồn được lưu cùng mô hình.

In [ ]:
FEATURE_SOURCE = '#!/usr/bin/env python3\n"""\ncompphish_features.py — Extract the CompPhish (Goenka 2026, Mendeley 10.17632/fmbs4kp9wz.2)\n26-column lexical URL feature schema, so our VN data is cross-dataset compatible with CompPhish.\n\nSchema (exact order):\n  url,url_len,dom,dom_len,is_ip,tld,tld_len,subdom_cnt,letter_cnt,digit_cnt,special_cnt,\n  eq_cnt,qm_cnt,amp_cnt,dot_cnt,dash_cnt,under_cnt,letter_ratio,digit_ratio,spec_ratio,\n  is_https,slash_cnt,entropy,path_len,query_len,label\n\nRequires: tldextract (for correct multi-part TLDs like edu.au, com.vn). pip install tldextract\nUse via align_compphish.py, or import extract().\n"""\nfrom __future__ import annotations\nimport re\nfrom collections import Counter\nfrom math import log2\nfrom urllib.parse import urlparse\n\ntry:\n    import tldextract\n    _EXT = tldextract.TLDExtract(suffix_list_urls=())  # offline snapshot\nexcept ImportError:  # pragma: no cover\n    _EXT = None\n\nCOLUMNS = ["url", "url_len", "dom", "dom_len", "is_ip", "tld", "tld_len", "subdom_cnt",\n           "letter_cnt", "digit_cnt", "special_cnt", "eq_cnt", "qm_cnt", "amp_cnt",\n           "dot_cnt", "dash_cnt", "under_cnt", "letter_ratio", "digit_ratio", "spec_ratio",\n           "is_https", "slash_cnt", "entropy", "path_len", "query_len"]\n\n_IP_RE = re.compile(r"^\\d{1,3}(\\.\\d{1,3}){3}$")\n\n\ndef _entropy(s: str) -> float:\n    if not s:\n        return 0.0\n    n = len(s)\n    return -sum((c / n) * log2(c / n) for c in Counter(s).values())\n\n\ndef extract_features(url: str) -> dict:\n    """Return the 25 feature columns (label added by caller).\n\n    NOTE: all length/count/ratio/entropy features are computed on the *scheme-stripped*\n    URL (``core``) so the stored http:// vs https:// prefix cannot leak into them. In our\n    collection benign URLs were stored with https and phishing with http, which made the\n    scheme a perfect (spurious) label predictor; ``is_https`` is kept for CompPhish schema\n    compatibility but is a known collection artifact — exclude it from modelling.\n    """\n    url = str(url).strip()\n    p = urlparse(url if re.match(r"^[a-z]+://", url, re.I) else "http://" + url)\n    host = (p.hostname or "").lower()\n    # scheme-stripped URL used for ALL lexical features (scheme-independent)\n    core = re.sub(r"^[a-z]+://", "", url, flags=re.I)\n\n    if _EXT is not None:\n        ext = _EXT(url if re.match(r"^[a-z]+://", url, re.I) else "http://" + url)\n        suffix = ext.suffix\n        dom = ext.top_domain_under_public_suffix or host\n        subdom = ext.subdomain\n    else:  # fallback (less accurate for multi-part TLDs)\n        parts = host.split(".")\n        suffix = parts[-1] if parts else ""\n        dom = ".".join(parts[-2:]) if len(parts) >= 2 else host\n        subdom = ".".join(parts[:-2]) if len(parts) > 2 else ""\n\n    url_len = len(core)\n    letter_cnt = sum(c.isalpha() for c in core)\n    digit_cnt = sum(c.isdigit() for c in core)\n    special_cnt = url_len - letter_cnt - digit_cnt\n    is_ip = 1 if _IP_RE.match(host) else 0\n    path_len = len(p.path.strip("/"))\n    return {\n        "url": url,\n        "url_len": url_len,\n        "dom": dom,\n        "dom_len": len(dom),\n        "is_ip": is_ip,\n        "tld": suffix,\n        "tld_len": len(suffix),\n        "subdom_cnt": len([s for s in subdom.split(".") if s]) if subdom else 0,\n        "letter_cnt": letter_cnt,\n        "digit_cnt": digit_cnt,\n        "special_cnt": special_cnt,\n        "eq_cnt": core.count("="),\n        "qm_cnt": core.count("?"),\n        "amp_cnt": core.count("&"),\n        "dot_cnt": core.count("."),\n        "dash_cnt": core.count("-"),\n        "under_cnt": core.count("_"),\n        "letter_ratio": letter_cnt / url_len if url_len else 0.0,\n        "digit_ratio": digit_cnt / url_len if url_len else 0.0,\n        "spec_ratio": special_cnt / url_len if url_len else 0.0,\n        "slash_cnt": core.count("/"),\n        "entropy": _entropy(core),\n        "path_len": path_len,\n        "query_len": len(p.query),\n    }\n\n'
exec(FEATURE_SOURCE)
FEATURE_NAMES = ['url_len', 'dom_len', 'is_ip', 'tld_len', 'subdom_cnt', 'letter_cnt', 'digit_cnt', 'special_cnt', 'eq_cnt', 'qm_cnt', 'amp_cnt', 'dot_cnt', 'dash_cnt', 'under_cnt', 'letter_ratio', 'digit_ratio', 'spec_ratio', 'slash_cnt', 'entropy', 'path_len', 'query_len']
assert "is_https" not in FEATURE_NAMES
def make_features(urls):
    result=[]
    for url in urls:
        parse_url_parts(url)  # cùng kiểm tra hợp lệ với đầu vào BERT/ứng dụng
        result.append(extract_features(url))
    out=pd.DataFrame(result)[FEATURE_NAMES].astype("float32")
    if not np.isfinite(out.to_numpy()).all():
        raise ValueError("Đặc trưng có giá trị thiếu/vô hạn.")
    return out
X={name:make_features(df.url.tolist()) for name,df in frames.items()}
y={name:df.labels.to_numpy(dtype=np.int64) for name,df in frames.items()}
display(X["train"].head())
display(X["train"].describe().T)

# Đối chiếu cách tính với đặc trưng lưu trong gói dữ liệu, không lấy nhãn làm đầu vào.
reference_features=pd.read_csv(PHISHVN_DIR/"data/vn_compphish.csv",low_memory=False)
reference_features["url_key"]=reference_features.url.map(format_url)
reference_features=reference_features.drop_duplicates("url_key").set_index("url_key")
sample=frames["train"].sample(min(500,len(frames["train"])),random_state=CFG["SEED"])
available=sample[sample.model_input.isin(reference_features.index)]
if available.empty:
    raise ValueError("Không đối chiếu được URL với bảng đặc trưng tham chiếu.")
computed=make_features(available.url.tolist()).to_numpy()
expected=reference_features.loc[available.model_input,FEATURE_NAMES].to_numpy(dtype=np.float32)
match=np.isclose(computed,expected,rtol=2e-5,atol=2e-5)
feature_audit={"rows":len(available),"fraction_matching":float(match.mean()),
    "different_columns":[name for j,name in enumerate(FEATURE_NAMES) if not match[:,j].all()]}
print("Đối chiếu đặc trưng:",feature_audit)
if not match.all():
    raise ValueError("Cách sinh đặc trưng khác bản tham chiếu; kiểm tra PSL/chuẩn hóa trước khi huấn luyện.")

## 6. Chỉ số và ngưỡng cảnh báo

Ngưỡng cảnh báo `tW`: tối đa F2 trên validation trong các ngưỡng có Precision ≥ 0,80.
Nếu không có ngưỡng đạt điều kiện, tối đa F2 và ghi rõ điều kiện chưa đạt.
F2 ưu tiên Recall hơn Precision để giảm bỏ sót. Ngưỡng cao `tH` cần Precision ≥ 0,95,
ít nhất 30 dự đoán dương và `tH ≥ tW`; không đạt thì tắt riêng mức cao từ URL.

Ngưỡng, cấu hình và mô hình đều chọn từ validation. Test được dùng sau khi đã khóa lựa chọn.
`average_precision` là AP, thường được báo cáo cùng tên PR-AUC; không dùng tích phân hình thang.
Các score chưa được hiệu chỉnh thành xác suất lừa đảo ngoài thực tế.

In [ ]:
def positive_scores(logits):
    if isinstance(logits,tuple):logits=logits[0]
    values=np.asarray(logits,dtype=np.float64)
    values=np.exp(values-values.max(axis=1,keepdims=True))
    return values[:,1]/values.sum(axis=1)

def binary_metrics(labels,scores,threshold=.5):
    labels=np.asarray(labels,dtype=int);scores=np.asarray(scores,dtype=float)
    pred=(scores>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(labels,pred,labels=[0,1]).ravel()
    both=len(np.unique(labels))==2
    return {"accuracy":float(accuracy_score(labels,pred)),
        "precision":float(precision_score(labels,pred,zero_division=0)),
        "recall":float(recall_score(labels,pred,zero_division=0)),
        "f1":float(f1_score(labels,pred,zero_division=0)),
        "f2":float(fbeta_score(labels,pred,beta=2,zero_division=0)),
        "roc_auc":float(roc_auc_score(labels,scores)) if both else None,
        "average_precision":float(average_precision_score(labels,scores)) if both else None,
        "false_positive_rate":float(fp/(fp+tn)) if fp+tn else None,
        "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)}

def choose_thresholds(labels,scores):
    labels=np.asarray(labels);scores=np.asarray(scores)
    if set(labels)!={0,1}:raise ValueError("Validation cần cả hai lớp.")
    precision,recall,thresholds=precision_recall_curve(labels,scores)
    precision=precision[:-1];recall=recall[:-1]
    f2=5*precision*recall/np.maximum(4*precision+recall,1e-12)
    feasible=precision>=CFG["MIN_WARNING_PRECISION"]
    pool=np.flatnonzero(feasible) if feasible.any() else np.arange(len(thresholds))
    best=pool[np.isclose(f2[pool],f2[pool].max(),atol=1e-12,rtol=0)]
    chosen=min(best,key=lambda j:abs(float(thresholds[j])-.5))
    warning=float(thresholds[chosen])
    counts=len(scores)-np.searchsorted(np.sort(scores),thresholds,side="left")
    high_pool=np.flatnonzero((precision>=CFG["MIN_HIGH_PRECISION"]) &
                            (counts>=CFG["MIN_HIGH_COUNT"]) & (thresholds>=warning))
    high=None
    if len(high_pool):
        i=max(high_pool,key=lambda j:(float(recall[j]),float(precision[j]),-float(thresholds[j])))
        high=float(thresholds[i])
    return {"warning_threshold":warning,"high_threshold":high,
            "warning_precision_met":bool(feasible.any()),"selected_on":"validation",
            "validation_metrics":binary_metrics(labels,scores,warning)}

def selection_score(settings):
    return settings["validation_metrics"]["f2"]-(0 if settings["warning_precision_met"] else 1)

fingerprint_config={k:v for k,v in CFG.items() if k not in {
    "WORK_DIR","DRIVE_DIR","USE_DRIVE","HF_CACHE_DIR","LOCAL_DATA_ZIP","ALLOW_CPU","RESUME_TRAINING"}}
fingerprint_config["pipeline_version"]="urltrain-2.1"
fingerprint_config["package_versions"]=VERSIONS
if CFG["RUN_BERT"]:
    fingerprint_config["bert_runtime"]={"gpu":HARDWARE["gpu"],
        "bf16":bool(HAS_CUDA and torch.cuda.is_bf16_supported()),
        "auto_batch":64 if VRAM_GB>=32 else 32 if VRAM_GB>=20 else 16}
fingerprint_config["feature_sha256"]=hashlib.sha256(FEATURE_SOURCE.encode()).hexdigest()
fingerprint_config["preprocessing_sha256"]=hashlib.sha256(PREPROCESSING_SOURCE.encode()).hexdigest()
split_digest=hashlib.sha256()
for name,df in frames.items():
    split_digest.update(name.encode());split_digest.update("\n".join(df.id).encode())
fingerprint_config["used_split_ids_sha256"]=split_digest.hexdigest()
fingerprint=hashlib.sha256(json.dumps(fingerprint_config,sort_keys=True).encode()).hexdigest()[:12]
RUN_ROOT=ROOT/("check_only" if CFG["SMOKE_TEST"] else "experiments")/fingerprint
RUNS=RUN_ROOT/"runs";RESULTS=RUN_ROOT/"results"
RUNS.mkdir(parents=True,exist_ok=True);RESULTS.mkdir(exist_ok=True)
registry=[]

def write_json(path,payload):
    path=Path(path);tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(payload,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
    tmp.replace(path)

def register_run(run_dir,family,seed,kind,parameters,val_scores,train_seconds):
    settings=choose_thresholds(y["val"],val_scores)
    spec={"run_id":run_dir.name,"family":family,"seed":int(seed),"kind":kind,
        "parameters":parameters,"train_seconds":train_seconds,"thresholds":settings,
        "validation_at_05":binary_metrics(y["val"],val_scores,.5),
        "selection_score":selection_score(settings),"fingerprint":fingerprint,
        "feature_names":FEATURE_NAMES,"smoke_test":CFG["SMOKE_TEST"],
        "hardware":HARDWARE,"versions":VERSIONS,
        "base_model_id":CFG["MODEL_ID"] if kind=="bert" else None}
    np.save(run_dir/"validation_scores.npy",val_scores)
    pd.DataFrame({"id":frames["val"].id,"label":y["val"],"score":val_scores}).to_csv(run_dir/"validation_predictions.csv",index=False)
    write_json(run_dir/"run.json",spec) # ghi cuối cùng: chỉ tái sử dụng lượt đã hoàn thành
    registry[:]=[item for item in registry if item["run_id"]!=spec["run_id"]]+[spec]
    return spec

def already_done(run_dir):
    marker=run_dir/"run.json"
    if CFG["RESUME_TRAINING"] and marker.is_file():
        spec=json.loads(marker.read_text())
        if spec["fingerprint"]!=fingerprint:raise ValueError("Sai fingerprint khi khôi phục.")
        registry[:]=[item for item in registry if item["run_id"]!=spec["run_id"]]+[spec]
        print("Dùng kết quả đã hoàn thành:",run_dir.name);return True
    return False

write_json(RUN_ROOT/"config.json",{"config":CFG,"fingerprint":fingerprint,
    "split_digest":split_digest.hexdigest(),"versions":VERSIONS,"hardware":HARDWARE,
    "cleaning":cleaning_log,"time_audit":time_audit,"feature_audit":feature_audit})
for name,df in frames.items():
    df[["id","domain_group","labels"]].to_csv(RESULTS/f"used_ids_{name}.csv",index=False)
print("Thư mục lần chạy:",RUN_ROOT)

## 7. Huấn luyện các mô hình ML

LogReg và RF dùng cấu hình tham chiếu từ PhishVN release `dib-113195`, trên cùng 21
đặc trưng đã đối chiếu. Đây là **so sánh trong giao thức đã khóa của notebook**:
không tự chép hoặc coi F1 công bố của một phép chia khác là cùng kết quả.
Ngoài ngưỡng đã chọn, notebook lưu kết quả ngưỡng 0,5 để tiện đối chiếu.
Mã gốc hỗ trợ cả BASIC9 và CompPhish21; notebook chọn CompPhish21 cho mọi mô hình ML.
Để tái lập một bảng cụ thể trong bài báo, cần đối chiếu cả schema, tập chia và tham số của bảng đó.

LightGBM/XGBoost thử hai cấu hình nhỏ, early stopping theo validation AP.
RF và LightGBM chạy CPU; XGBoost bắt buộc dùng CUDA nếu đã phát hiện GPU.

In [ ]:
if CFG["RUN_TREES"]:
    threads=int(CFG["CPU_THREADS"])
    seeds=[CFG["SEED"]] if CFG["SMOKE_TEST"] else CFG["ML_SEEDS"]
    configs=[("LogReg_ref", "logreg", {}),
             ("RandomForest_ref", "rf", {"n_estimators":300}),
             ("LightGBM_31", "lightgbm", {"num_leaves":31,"min_child_samples":30}),
             ("LightGBM_63", "lightgbm", {"num_leaves":63,"min_child_samples":60}),
             ("XGBoost_d4", "xgboost", {"max_depth":4,"min_child_weight":3}),
             ("XGBoost_d6", "xgboost", {"max_depth":6,"min_child_weight":6})]
    if CFG["SMOKE_TEST"]:
        configs=[item for item in configs if item[0] not in {"LightGBM_63","XGBoost_d6"}]
    for family,kind,params in configs:
        for seed in (seeds[:1] if kind=="logreg" else seeds):
            run_dir=RUNS/f"{family}_s{seed}";run_dir.mkdir(exist_ok=True)
            if already_done(run_dir):continue
            started=time.perf_counter()
            if kind=="logreg":
                model=LogisticRegression(max_iter=1000,class_weight="balanced",random_state=seed)
                model.fit(X["train"],y["train"])
                val_scores=model.predict_proba(X["val"])[:,1]
                joblib.dump(model,run_dir/"model.joblib",compress=3)
            elif kind=="rf":
                model=RandomForestClassifier(n_estimators=12 if CFG["SMOKE_TEST"] else params["n_estimators"],
                    class_weight="balanced",n_jobs=threads,random_state=seed)
                model.fit(X["train"],y["train"])
                val_scores=model.predict_proba(X["val"])[:,1]
                joblib.dump(model,run_dir/"model.joblib",compress=3)
            elif kind=="lightgbm":
                model=lgb.LGBMClassifier(n_estimators=12 if CFG["SMOKE_TEST"] else CFG["BOOST_ROUNDS"],
                    learning_rate=.03,max_depth=-1,class_weight="balanced",reg_lambda=1.,
                    colsample_bytree=.9,subsample=.9,subsample_freq=1,verbosity=-1,
                    n_jobs=threads,random_state=seed,device_type="cpu",metric="average_precision",**params)
                model.fit(X["train"],y["train"],eval_set=[(X["val"],y["val"])],
                    eval_metric="average_precision",callbacks=[lgb.early_stopping(
                        3 if CFG["SMOKE_TEST"] else CFG["EARLY_STOPPING"],verbose=False)])
                val_scores=model.predict_proba(X["val"])[:,1]
                model.booster_.save_model(str(run_dir/"model.txt"))
            elif kind=="xgboost":
                pos=int(y["train"].sum());neg=len(y["train"])-pos
                model=xgb.XGBClassifier(n_estimators=12 if CFG["SMOKE_TEST"] else CFG["BOOST_ROUNDS"],
                    learning_rate=.03,tree_method="hist",device="cuda" if HAS_CUDA else "cpu",
                    scale_pos_weight=neg/pos,subsample=.9,colsample_bytree=.9,reg_lambda=1.,
                    eval_metric="aucpr",early_stopping_rounds=3 if CFG["SMOKE_TEST"] else CFG["EARLY_STOPPING"],
                    n_jobs=threads,random_state=seed,**params)
                model.fit(X["train"],y["train"],eval_set=[(X["val"],y["val"])],verbose=False)
                if HAS_CUDA:
                    device=json.loads(model.get_booster().save_config())["learner"]["generic_param"]["device"]
                    if not device.startswith("cuda"):
                        raise RuntimeError("XGBoost đã rơi về CPU. Kiểm tra CUDA/driver trước khi chạy tiếp.")
                val_scores=model.predict_proba(X["val"])[:,1]
                model.save_model(run_dir/"model.ubj")
            elapsed=time.perf_counter()-started
            spec=register_run(run_dir,family,seed,kind,params,val_scores,elapsed)
            print(f'{run_dir.name}: val F2={spec["thresholds"]["validation_metrics"]["f2"]:.4f}; '
                  f'Precision đạt điều kiện={spec["thresholds"]["warning_precision_met"]}; {elapsed:.1f}s')
            del model;gc.collect()
else:
    print("Bỏ qua mô hình ML theo RUN_TREES=False.")

## 8. Fine-tune DomURLs_BERT

Tokenizer và encoder dùng cùng revision đã ghim. Đồng bộ PAD ID vì tokenizer gốc
và config có giá trị khác nhau; giới hạn độ dài theo cấu hình encoder.
Batch tự chọn 64 với VRAM ≥ 32 GB, 32 với ≥ 20 GB và 16 với GPU nhỏ hơn.
Dùng BF16 khi GPU hỗ trợ, nếu không thì FP16; SDPA và TF32 giúp tận dụng GPU.
Nếu hết VRAM, giảm `BERT_BATCH_SIZE`; gradient accumulation duy trì batch hiệu dụng.
Cuối mỗi lượt, tính lại validation bằng FP32 từ mô hình đã lưu để chọn ngưỡng
cùng chế độ suy luận với mã ứng dụng; không lấy ngưỡng từ score BF16 rồi áp cho score FP32.

In [ ]:
if CFG["RUN_BERT"]:
    tokenizer=AutoTokenizer.from_pretrained(CFG["MODEL_ID"],revision=CFG["MODEL_REVISION"],
        cache_dir=CFG["HF_CACHE_DIR"],use_fast=True,trust_remote_code=False)
    encoder_config=AutoConfig.from_pretrained(CFG["MODEL_ID"],revision=CFG["MODEL_REVISION"],
        cache_dir=CFG["HF_CACHE_DIR"])
    MAX_LENGTH=min(CFG["BERT_MAX_LENGTH"],encoder_config.max_position_embeddings)
    if CFG["SMOKE_TEST"]:MAX_LENGTH=min(MAX_LENGTH,64)
    if tokenizer.pad_token_id is None or len(tokenizer)!=encoder_config.vocab_size:
        raise ValueError("Tokenizer và encoder không khớp.")
    for token in ["[DOMAIN]","[PATH]","[IP]","[IPv6]"]:
        if tokenizer.convert_tokens_to_ids(token)==tokenizer.unk_token_id:raise ValueError(f"Thiếu {token}")
    tokenized=DatasetDict()
    for name,df in frames.items():
        ds=Dataset.from_dict({"model_input":df.model_input.tolist(),"labels":y[name].tolist()})
        tokenized[name]=ds.map(lambda b:tokenizer(b["model_input"],max_length=MAX_LENGTH,
            truncation=True,padding=False),batched=True,remove_columns=["model_input"],desc=f"Tokenize {name}")
    train_sample=frames["train"].sample(min(2000,len(frames["train"])),random_state=CFG["SEED"])
    lengths=np.asarray(tokenizer(train_sample.model_input.tolist(),truncation=False,return_length=True)["length"])
    token_audit={"max_length":MAX_LENGTH,"p95":float(np.quantile(lengths,.95)),
        "truncated_fraction":float((lengths>MAX_LENGTH).mean()),"pad_token_id":tokenizer.pad_token_id}
    print("Token audit:",token_audit)
    collator=DataCollatorWithPadding(tokenizer,pad_to_multiple_of=8 if HAS_CUDA else None)
    automatic_batch=64 if VRAM_GB>=32 else 32 if VRAM_GB>=20 else 16
    BERT_BATCH=int(CFG["BERT_BATCH_SIZE"] or automatic_batch)
    if CFG["SMOKE_TEST"]:BERT_BATCH=2
    ACCUM=1 if CFG["SMOKE_TEST"] else max(1,math.ceil(CFG["BERT_EFFECTIVE_BATCH"]/BERT_BATCH))
    GRAD_CKPT=(VRAM_GB<24) if CFG["BERT_GRAD_CHECKPOINTING"] is None else CFG["BERT_GRAD_CHECKPOINTING"]
    USE_BF16=HAS_CUDA and torch.cuda.is_bf16_supported()
    class_weights=torch.tensor(compute_class_weight("balanced",classes=np.array([0,1]),y=y["train"]),dtype=torch.float32)

    class WeightedTrainer(Trainer):
        def __init__(self,*args,class_weights=None,**kwargs):
            super().__init__(*args,**kwargs);self.class_weights=class_weights
            self.model_accepts_loss_kwargs=False
        def compute_loss(self,model,inputs,return_outputs=False,num_items_in_batch=None):
            outputs=model(**{k:v for k,v in inputs.items() if k!="labels"})
            loss=torch.nn.functional.cross_entropy(outputs.logits.float(),inputs["labels"],
                weight=self.class_weights.to(outputs.logits.device))
            return (loss,outputs) if return_outputs else loss

    def bert_metrics(prediction):
        scores=positive_scores(prediction.predictions)
        settings=choose_thresholds(prediction.label_ids,scores)
        return {**settings["validation_metrics"],"selection_score":selection_score(settings)}

    seeds=[CFG["SEED"]] if CFG["SMOKE_TEST"] else CFG["BERT_SEEDS"]
    for seed in seeds:
        run_dir=RUNS/f"DomURLs_BERT_s{seed}";run_dir.mkdir(exist_ok=True)
        if already_done(run_dir):continue
        set_seed(seed)
        if HAS_CUDA:
            torch.set_float32_matmul_precision("high")
            torch.backends.cuda.matmul.allow_tf32=True
        conf=AutoConfig.from_pretrained(CFG["MODEL_ID"],revision=CFG["MODEL_REVISION"],cache_dir=CFG["HF_CACHE_DIR"])
        conf.num_labels=2;conf.id2label={0:"BENIGN",1:"PHISHING"};conf.label2id={"BENIGN":0,"PHISHING":1}
        conf.pad_token_id=tokenizer.pad_token_id
        model,loading=AutoModelForSequenceClassification.from_pretrained(
            CFG["MODEL_ID"],revision=CFG["MODEL_REVISION"],config=conf,cache_dir=CFG["HF_CACHE_DIR"],
            use_safetensors=True,trust_remote_code=False,attn_implementation="sdpa",output_loading_info=True)
        bad=[k for k in loading["missing_keys"] if not k.startswith("classifier.")]
        if bad or loading.get("unexpected_keys") or loading.get("mismatched_keys"):
            raise ValueError(f"Không tải đúng trọng số encoder: {loading}")
        checkpoints=run_dir/"checkpoints";checkpoints.mkdir(exist_ok=True)
        args=TrainingArguments(output_dir=str(checkpoints),num_train_epochs=CFG["BERT_EPOCHS"],
            max_steps=2 if CFG["SMOKE_TEST"] else -1,learning_rate=CFG["BERT_LR"],
            per_device_train_batch_size=BERT_BATCH,per_device_eval_batch_size=max(8,BERT_BATCH),
            gradient_accumulation_steps=ACCUM,gradient_checkpointing=GRAD_CKPT,
            eval_strategy="epoch",save_strategy="epoch",save_total_limit=2,load_best_model_at_end=True,
            metric_for_best_model="selection_score",greater_is_better=True,
            fp16=HAS_CUDA and not USE_BF16,bf16=USE_BF16,tf32=HAS_CUDA and torch.cuda.get_device_capability(0)[0]>=8,
            optim="adamw_torch",weight_decay=.01,warmup_ratio=.1,max_grad_norm=1.,logging_steps=25,
            dataloader_num_workers=0,seed=seed,data_seed=seed,report_to=[],push_to_hub=False,
            save_safetensors=True,use_cpu=not HAS_CUDA)
        trainer=WeightedTrainer(model=model,args=args,train_dataset=tokenized["train"],
            eval_dataset=tokenized["val"],processing_class=tokenizer,data_collator=collator,
            compute_metrics=bert_metrics,class_weights=class_weights,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
        resume=get_last_checkpoint(str(checkpoints)) if CFG["RESUME_TRAINING"] else None
        print("Fine-tune",seed,"| batch",BERT_BATCH,"| accumulation",ACCUM,
              "| BF16",USE_BF16,"| gradient checkpointing",GRAD_CKPT,"| resume",resume)
        if HAS_CUDA:torch.cuda.reset_peak_memory_stats()
        start=time.perf_counter();train_result=trainer.train(resume_from_checkpoint=resume)
        elapsed=time.perf_counter()-start
        trainer.save_model(str(run_dir));tokenizer.save_pretrained(str(run_dir))
        pd.DataFrame(trainer.state.log_history).to_csv(run_dir/"training_history.csv",index=False)
        params={"max_length":MAX_LENGTH,"learning_rate":CFG["BERT_LR"],"epochs":CFG["BERT_EPOCHS"],
            "batch":BERT_BATCH,"accumulation":ACCUM,"bf16":USE_BF16,"gradient_checkpointing":GRAD_CKPT,
            "model_revision":CFG["MODEL_REVISION"],"best_checkpoint":trainer.state.best_model_checkpoint,
            "peak_vram_gb":torch.cuda.max_memory_allocated()/1024**3 if HAS_CUDA else None,
            "class_weights":class_weights.tolist(),"token_audit":token_audit,
            "final_validation_precision":"float32_no_tf32"}
        del trainer,model;gc.collect()
        if HAS_CUDA:torch.cuda.empty_cache()
        # Nạp độc lập, không dùng forward đã được Accelerate bọc autocast.
        from torch.utils.data import DataLoader
        deployed_model=AutoModelForSequenceClassification.from_pretrained(run_dir,
            local_files_only=True,use_safetensors=True,attn_implementation="sdpa").float()
        device="cuda" if HAS_CUDA else "cpu"
        if HAS_CUDA:
            torch.set_float32_matmul_precision("highest")
            torch.backends.cuda.matmul.allow_tf32=False
        deployed_model.to(device).eval();val_parts=[]
        loader=DataLoader(tokenized["val"],batch_size=max(8,BERT_BATCH),shuffle=False,collate_fn=collator)
        with torch.inference_mode():
            for batch in loader:
                inputs={k:v.to(device) for k,v in batch.items() if k!="labels"}
                logits=deployed_model(**inputs).logits.float().cpu().numpy()
                val_parts.append(positive_scores(logits))
        val_scores=np.concatenate(val_parts)
        del deployed_model,loader;gc.collect()
        if HAS_CUDA:torch.cuda.empty_cache()
        spec=register_run(run_dir,"DomURLs_BERT",seed,"bert",params,val_scores,elapsed)
        print("Đã lưu",run_dir,"| val F2",spec["thresholds"]["validation_metrics"]["f2"])
else:
    print("Bỏ qua BERT theo RUN_BERT=False.")

## 9. Khóa mô hình và ngưỡng bằng validation

Chọn lượt chạy có điều kiện Precision đạt trước, sau đó F2, AP và Precision trên validation.
Seed cũng được xem là một lựa chọn từ validation. Kết quả trung bình/độ lệch chuẩn theo
cấu hình được báo cáo riêng; mô hình triển khai là **một lượt đã chọn**, không phải ensemble.

In [ ]:
if not registry:raise ValueError("Chưa có mô hình hoàn thành.")
rows=[]
for spec in registry:
    rows.append({"run_id":spec["run_id"],"family":spec["family"],"seed":spec["seed"],
        "kind":spec["kind"],"selection_score":spec["selection_score"],
        "precision_met":spec["thresholds"]["warning_precision_met"],
        "warning_threshold":spec["thresholds"]["warning_threshold"],
        "high_threshold":spec["thresholds"]["high_threshold"],
        **spec["thresholds"]["validation_metrics"]})
val_table=pd.DataFrame(rows).sort_values(["selection_score","average_precision","precision"],ascending=False)
display(val_table)
val_table.to_csv(RESULTS/"validation_leaderboard.csv",index=False)
winner_id=val_table.iloc[0].run_id
winner=next(spec for spec in registry if spec["run_id"]==winner_id)
LOCK={"selected_run_id":winner_id,"selection_data":"validation","fingerprint":fingerprint,
      "warning_threshold":winner["thresholds"]["warning_threshold"],
      "high_threshold":winner["thresholds"]["high_threshold"],"smoke_test":CFG["SMOKE_TEST"]}
write_json(RESULTS/"selection_lock.json",LOCK)
print("Mô hình đã chọn:",winner_id,"| Không thay đổi lựa chọn theo kết quả test.")

## 10. Mã nạp mô hình và dự đoán dùng chung

Cùng hàm xử lý được dùng cho đánh giá và ứng dụng. Các mô hình cây chạy suy luận CPU
được dù đã train trên GPU. BERT tự dùng CUDA nếu có. Mức LOW chỉ có nghĩa chưa vượt ngưỡng
theo mô hình, không có nghĩa URL được xác minh an toàn.

In [ ]:
INFERENCE_SOURCE = '\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom url_preprocessing import format_url, parse_url_parts\nfrom url_features import extract_features\n\nclass URLRiskPredictor:\n    def __init__(self,model_dir,device=None):\n        self.directory=Path(model_dir)\n        self.spec=json.loads((self.directory/"run.json").read_text(encoding="utf-8"))\n        self.kind=self.spec["kind"]\n        self.settings=self.spec["thresholds"]\n        self.tW=self.settings["warning_threshold"]\n        self.tH=self.settings["high_threshold"]\n        self.features=self.spec["feature_names"]\n        if self.kind in {"logreg","rf"}:\n            import joblib\n            self.model=joblib.load(self.directory/"model.joblib")\n        elif self.kind=="lightgbm":\n            import lightgbm as lgb\n            self.model=lgb.Booster(model_file=str(self.directory/"model.txt"))\n        elif self.kind=="xgboost":\n            import xgboost as xgb\n            self.model=xgb.XGBClassifier()\n            self.model.load_model(self.directory/"model.ubj")\n            self.model.set_params(device="cpu")\n        elif self.kind=="bert":\n            import torch\n            from transformers import AutoModelForSequenceClassification,AutoTokenizer\n            self.device=device or ("cuda" if torch.cuda.is_available() else "cpu")\n            if self.device.startswith("cuda"):\n                torch.set_float32_matmul_precision("highest")\n                torch.backends.cuda.matmul.allow_tf32=False\n            self.tokenizer=AutoTokenizer.from_pretrained(self.directory,local_files_only=True)\n            self.model=AutoModelForSequenceClassification.from_pretrained(\n                self.directory,local_files_only=True,use_safetensors=True,attn_implementation="sdpa").to(self.device).eval()\n            self.max_length=self.spec["parameters"]["max_length"]\n        else:raise ValueError(f"Loại mô hình không được hỗ trợ: {self.kind}")\n\n    def predict(self,urls,batch_size=32):\n        if isinstance(urls,str):urls=[urls]\n        urls=list(urls)\n        if batch_size<1:raise ValueError("batch_size cần lớn hơn 0.")\n        if not urls:return []\n        for url in urls:parse_url_parts(url)\n        truncated=[False]*len(urls)\n        if self.kind=="bert":\n            import torch\n            texts=[format_url(u) for u in urls];scores=[]\n            for start in range(0,len(texts),batch_size):\n                chunk=texts[start:start+batch_size]\n                lengths=self.tokenizer(chunk,truncation=False,return_length=True)["length"]\n                batch=self.tokenizer(chunk,padding=True,truncation=True,max_length=self.max_length,\n                    return_tensors="pt").to(self.device)\n                with torch.inference_mode():\n                    logits=self.model(**batch).logits.float().cpu().numpy().astype(np.float64)\n                    exp=np.exp(logits-logits.max(axis=1,keepdims=True))\n                    scores.extend((exp[:,1]/exp.sum(axis=1)).tolist())\n                truncated[start:start+len(chunk)]=[n>self.max_length for n in lengths]\n        else:\n            X=pd.DataFrame([extract_features(u) for u in urls])[self.features].astype("float32")\n            if not np.isfinite(X.to_numpy()).all():raise ValueError("Đặc trưng không hợp lệ.")\n            if self.kind=="lightgbm":scores=self.model.predict(X).tolist()\n            else:scores=self.model.predict_proba(X)[:,1].tolist()\n        output=[]\n        for url,score,cut in zip(urls,scores,truncated):\n            level="HIGH" if self.tH is not None and score>=self.tH else "WARNING" if score>=self.tW else "LOW"\n            output.append({"url":url,"score":float(score),"risk_level":level,\n                "is_warning":int(score>=self.tW),"warning_threshold":self.tW,\n                "high_threshold":self.tH,"truncated":bool(cut)})\n        return output\n'
MODULES=RUN_ROOT/"inference_code";MODULES.mkdir(exist_ok=True)
(MODULES/"url_features.py").write_text(FEATURE_SOURCE,encoding="utf-8")
(MODULES/"url_preprocessing.py").write_text(PREPROCESSING_SOURCE,encoding="utf-8")
(MODULES/"predict_url.py").write_text(INFERENCE_SOURCE,encoding="utf-8")
sys.path.insert(0,str(MODULES.resolve()))
for module in ["url_features","url_preprocessing","predict_url"]:sys.modules.pop(module,None)
from predict_url import URLRiskPredictor

## 11. Đánh giá test sau khi đã khóa lựa chọn

Lưu bảng so sánh mọi lượt chạy, trung bình/độ lệch chuẩn qua seed, kết quả từng URL,
và phân tích theo `tier`/`source`. Không dùng thứ tự test để đổi mô hình đã khóa.
Chỉ mô hình đã chọn được vẽ confusion matrix và đường Precision–Recall ở phần tổng hợp.

In [ ]:
stored_lock=json.loads((RESULTS/"selection_lock.json").read_text())
assert stored_lock==LOCK and stored_lock["fingerprint"]==fingerprint
test_rows=[];group_rows=[];winner_predictions=None
for spec in registry:
    predictor=URLRiskPredictor(RUNS/spec["run_id"])
    batch=8 if CFG["SMOKE_TEST"] else 64
    outputs=predictor.predict(frames["test"].url.tolist(),batch_size=batch)
    scores=np.asarray([o["score"] for o in outputs])
    metrics=binary_metrics(y["test"],scores,spec["thresholds"]["warning_threshold"])
    at05=binary_metrics(y["test"],scores,.5)
    test_rows.append({"run_id":spec["run_id"],"family":spec["family"],"seed":spec["seed"],
        "selected_by_validation":spec["run_id"]==winner_id,**metrics,
        "f1_at_05":at05["f1"],"recall_at_05":at05["recall"],"precision_at_05":at05["precision"]})
    result=frames["test"][["id","url","labels","source","tier","domain_group"]].copy()
    result["score"]=scores;result["prediction"]=[o["is_warning"] for o in outputs]
    result["risk_level"]=[o["risk_level"] for o in outputs]
    result["correct"]=result.labels.eq(result.prediction)
    result.to_csv(RESULTS/f'test_predictions_{spec["run_id"]}.csv',index=False)
    for column in ["tier","source"]:
        for value,df in result.groupby(column):
            group_rows.append({"run_id":spec["run_id"],"group_by":column,"group":value,
                "rows":len(df),"benign":int((df.labels==0).sum()),"phishing":int((df.labels==1).sum()),
                **binary_metrics(df.labels,df.score,spec["thresholds"]["warning_threshold"])})
    core=result[result.tier.isin(["gold","silver"])]
    if len(core):group_rows.append({"run_id":spec["run_id"],"group_by":"subset","group":"gold_silver",
        "rows":len(core),"benign":int((core.labels==0).sum()),"phishing":int((core.labels==1).sum()),
        **binary_metrics(core.labels,core.score,spec["thresholds"]["warning_threshold"])})
    if spec["run_id"]==winner_id:winner_predictions=result.copy()
    del predictor;gc.collect()
    if HAS_CUDA:torch.cuda.empty_cache()
test_table=pd.DataFrame(test_rows)
test_table.to_csv(RESULTS/"test_comparison.csv",index=False)
pd.DataFrame(group_rows).to_csv(RESULTS/"test_by_source_tier.csv",index=False)
seed_summary=test_table.groupby("family")[["precision","recall","f1","f2","average_precision"]].agg(["mean","std","count"])
seed_summary.to_csv(RESULTS/"test_seed_summary.csv")
print("KIỂM TRA MÃ — KHÔNG DÙNG KẾT QUẢ CHO BÁO CÁO" if CFG["SMOKE_TEST"] else "TEST SAU KHI KHÓA MÔ HÌNH")
display(test_table);display(seed_summary)
display(pd.DataFrame(group_rows).query("run_id == @winner_id"))
print("Mô hình triển khai vẫn là:",winner_id)

In [ ]:
chosen=winner_predictions
print(classification_report(chosen.labels,chosen.prediction,labels=[0,1],
    target_names=["BENIGN","PHISHING/SCAM"],digits=4,zero_division=0))
fig,axes=plt.subplots(1,2,figsize=(11,4))
ConfusionMatrixDisplay(confusion_matrix(chosen.labels,chosen.prediction,labels=[0,1]),
    display_labels=["BENIGN","PHISHING"]).plot(ax=axes[0],colorbar=False)
p,r,_=precision_recall_curve(chosen.labels,chosen.score)
axes[1].plot(r,p);axes[1].set(xlabel="Recall",ylabel="Precision",title="Precision–Recall trên test")
fig.suptitle(("SMOKE CHECK — " if CFG["SMOKE_TEST"] else "")+winner_id)
fig.tight_layout();fig.savefig(RESULTS/"selected_model_evaluation.png",dpi=170);plt.show()
errors=chosen[~chosen.correct]
errors.to_csv(RESULTS/"selected_model_errors.csv",index=False)
display(errors[["url","labels","prediction","score","source","tier"]].head(20))

# Biểu đồ validation giúp đọc việc lựa chọn, không chọn theo cột test.
fig,ax=plt.subplots(figsize=(10,max(4,len(val_table)*.25)))
ax.barh(val_table.run_id.iloc[::-1],val_table.f2.iloc[::-1]);ax.set(xlabel="Validation F2",title="Kết quả validation")
fig.tight_layout();fig.savefig(RESULTS/"validation_comparison.png",dpi=160);plt.show()

## 12. Xuất mô hình đã chọn, kiểm tra nạp lại và đo thời gian

Gói xuất chứa một mô hình đã chọn bằng validation, mã sinh đặc trưng/xử lý URL,
ngưỡng, phiên bản, giấy phép nguồn và các bảng kết quả. Checkpoint/optimizer và các
lượt train khác vẫn ở thư mục làm việc, không đưa vào ZIP.

In [ ]:
EXPORT_DIR=RUN_ROOT/f"url_model_export_{winner_id}";EXPORT_DIR.mkdir(exist_ok=True)
source=RUNS/winner_id
files=["run.json","model.joblib","model.txt","model.ubj","config.json","model.safetensors",
       "tokenizer.json","tokenizer_config.json","special_tokens_map.json","vocab.txt","training_history.csv"]
for name in files:
    if (source/name).is_file():shutil.copy2(source/name,EXPORT_DIR/name)
for name in ["url_features.py","url_preprocessing.py","predict_url.py"]:
    shutil.copy2(MODULES/name,EXPORT_DIR/name)
shutil.copytree(RESULTS,EXPORT_DIR/"results",dirs_exist_ok=True)
shutil.copy2(RUN_ROOT/"config.json",EXPORT_DIR/"experiment_config.json")
for name in ["LICENSE","CITATION.cff"]:
    shutil.copy2(PHISHVN_DIR/name,EXPORT_DIR/("DATA_"+name))

In [ ]:
CODE_LICENSE = 'MIT License\n\nCopyright (c) 2026 Thai Nguyen Vu (UTC2)\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n'
(EXPORT_DIR/"PHISHVN_CODE_LICENSE.txt").write_text(CODE_LICENSE,encoding="utf-8")
(EXPORT_DIR/"requirements.txt").write_text(
    "torch>=2.6\ntransformers==4.57.6\nxgboost==3.0.5\nlightgbm==4.6.0\n"
    "scikit-learn==1.7.2\npandas>=2.2,<3\nnumpy>=1.26,<3\ntldextract==5.3.0\njoblib>=1.3\n",encoding="utf-8")
manifest={"selected_run":winner,"lock":LOCK,"config":CFG,"versions":VERSIONS,"hardware":HARDWARE,
    "dataset":{"doi":"10.17632/b97hxbxtpd.4","sha256":CFG["DATA_SHA256"]},
    "test_results":test_table.to_dict("records"),"cleaning":cleaning_log,"time_audit":time_audit,
    "feature_audit":feature_audit,"used_split_digest":split_digest.hexdigest(),
    "reference":"https://github.com/vuthainguyen1602/phishvn/releases/tag/dib-113195"}
write_json(EXPORT_DIR/"manifest.json",manifest)
readme=f"""# URL phishing model

Selected run: {winner_id}. Selected using validation, never the test leaderboard.
smoke_test={CFG['SMOKE_TEST']}; DATA_SCOPE={CFG['DATA_SCOPE']}.
Data: Thai Nguyen Vu, PhishVN V4, https://doi.org/10.17632/b97hxbxtpd.4, CC BY 4.0.
Upstream: NCSC Tin Nhiem Mang, ChongLuaDao, OpenPhish, Tranco.
Lexical features and reference settings adapted from PhishVN release dib-113195 (MIT).
Encoder when used: https://huggingface.co/amahdaouy/DomURLs_BERT

Install requirements.txt in a suitable PyTorch CPU/CUDA environment. Run from this folder:

```python
from predict_url import URLRiskPredictor
predictor = URLRiskPredictor('.')
print(predictor.predict(['https://example.org/']))
```

Scores are uncalibrated model outputs. LOW does not certify that a URL is safe.
Inference treats URLs as text and does not visit their destinations.
The negative and positive classes follow the dataset labels, including scam URLs beyond credential phishing.
See manifest.json and results/ for actual metrics and model selection. Smoke-test metrics are not benchmark results.
"""
(EXPORT_DIR/"README.md").write_text(readme,encoding="utf-8")

# Nạp từ gói xuất, kiểm tra xác suất so với lần đánh giá đã lưu.
predictor=URLRiskPredictor(EXPORT_DIR)
sample_urls=frames["test"].url.head(8).tolist()
reloaded=predictor.predict(sample_urls,batch_size=8)
np.testing.assert_allclose([o["score"] for o in reloaded],winner_predictions.score.head(8).to_numpy(),rtol=3e-5,atol=3e-6)
print("Nạp lại mô hình: OK")
MY_URLS=["https://example.org/", "https://login.verify-account.example.test/verify?request=123"]
display(pd.DataFrame(predictor.predict(MY_URLS)))
timings=[]
for i in range(3):predictor.predict(sample_urls[:1])
for i in range(30):
    start=time.perf_counter();predictor.predict(sample_urls[i%len(sample_urls)])
    timings.append((time.perf_counter()-start)*1000)
latency={"requests":len(timings),"median_ms":float(np.median(timings)),
         "p95_ms":float(np.percentile(timings,95)),"includes_url_preprocessing":True,"model_load_excluded":True,
         "device":getattr(predictor,"device","cpu")}
write_json(EXPORT_DIR/"latency.json",latency);print("Thời gian dự đoán 1 URL:",latency)

## 13. Tải gói mô hình

ZIP chỉ chứa các tệp xuất dùng được độc lập. Nếu đã bật Drive, các lượt chạy và
checkpoint được giữ ở `DRIVE_DIR`; nếu không, hãy tải ZIP trước khi kết thúc phiên.

In [ ]:
MODEL_ZIP=RUN_ROOT/("url_model_check_only.zip" if CFG["SMOKE_TEST"] else "url_phishing_model.zip")
with zipfile.ZipFile(MODEL_ZIP,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=1) as package:
    for path in sorted(EXPORT_DIR.rglob("*")):
        rel=path.relative_to(EXPORT_DIR)
        if path.is_file() and not any(part.startswith(".") or part=="__pycache__" for part in rel.parts):
            package.write(path,arcname=str(rel))
print("Thư mục xuất:",EXPORT_DIR)
print("Gói mô hình:",MODEL_ZIP,"| MB:",round(MODEL_ZIP.stat().st_size/1024**2,1))

In [ ]:
# Chạy khi muốn tải gói mô hình về máy.
try:
    from google.colab import files
except ImportError:
    print("Ngoài Colab: tải tệp từ đường dẫn",MODEL_ZIP)
else:
    files.download(str(MODEL_ZIP))

## Nguồn và cách đọc kết quả

- [PhishVN v4 và giấy phép dữ liệu](https://data.mendeley.com/datasets/b97hxbxtpd/4).
- [Mã PhishVN, release dùng cho bài Data in Brief](https://github.com/vuthainguyen1602/phishvn/releases/tag/dib-113195).
- [DomURLs_BERT](https://huggingface.co/amahdaouy/DomURLs_BERT) và [bài nghiên cứu](https://arxiv.org/abs/2409.09143).
- [XGBoost 3.0 GPU](https://xgboost.readthedocs.io/en/release_3.0.0/gpu/index.html).

Đọc `validation_leaderboard.csv` để biết lý do chọn mô hình; `test_comparison.csv` và
`test_seed_summary.csv` để báo cáo kết quả; `test_by_source_tier.csv` để xem khác biệt
gold/silver/bronze; `selected_model_errors.csv` để phân tích bỏ sót/cảnh báo nhầm.
Nếu đổi cấu hình sau khi xem test, cần nêu đó là quá trình phát triển thêm, không coi
test cũ như một phép đo độc lập mới. GPU mạnh giúp chạy nhanh và nhiều seed, không bảo đảm F1 cao hơn.